In [1]:
import sys; sys.path.append("..")
import re
import shutil
from pathlib import Path
import pandas as pd
from sqlalchemy import text
from db import engine

ROOT = Path("..")
DATA = ROOT / "data"
INCOMING = DATA / "incoming"
PROCESSED = DATA / "processed"
SQL = ROOT / "sql"
PROCESSED.mkdir(exist_ok=True)
print("Connected. Files waiting in incoming:", len(list(INCOMING.glob("*.csv"))))

Connected. Files waiting in incoming: 7


In [2]:
def run_sql_file(path):
    script = Path(path).read_text()
    batches = [b.strip() for b in re.split(r"^\s*GO\s*$", script, flags=re.MULTILINE | re.IGNORECASE) if b.strip()]
    with engine.begin() as conn:
        for batch in batches:
            conn.execute(text(batch))
    print("Ran", Path(path).name, "-", len(batches), "batches")

In [3]:
rows_in_db = pd.read_sql("SELECT COUNT(*) AS n FROM raw.requests", engine)["n"][0]
first_run = rows_in_db == 0
print("Rows already in raw.requests:", rows_in_db)
print("First run?", first_run)

Rows already in raw.requests: 1260
First run? False


In [4]:
HISTORY_FILES = {
    "requests":    ["week_start"],
    "handle_time": [],
    "roster":      ["start_date", "end_date"],
    "events":      ["event_date"],
    "assumptions": ["review_date"],
}

week_loaded = None
rows_loaded = 0

if first_run:
    for table, date_cols in HISTORY_FILES.items():
        df = pd.read_csv(DATA / f"{table}.csv", parse_dates=date_cols)
        df.to_sql(table, engine, schema="raw", if_exists="append", index=False)
        print(f"Loaded {len(df):,} rows into raw.{table}")
        if table == "requests":
            rows_loaded = len(df)
            week_loaded = df["week_start"].max().date()
else:
    waiting = sorted(INCOMING.glob("*.csv"))
    if not waiting:
        print("No new files in incoming. Nothing to load.")
    else:
        next_file = waiting[0]
        df = pd.read_csv(next_file, parse_dates=["week_start"])
        df.to_sql("requests", engine, schema="raw", if_exists="append", index=False)
        shutil.move(str(next_file), str(PROCESSED / next_file.name))
        rows_loaded = len(df)
        week_loaded = df["week_start"].max().date()
        print(f"Loaded {next_file.name}: {rows_loaded} rows, then moved it to processed")

print("Week loaded:", week_loaded)

Loaded requests_2026-10-26.csv: 12 rows, then moved it to processed


Week loaded: 2026-10-26


In [5]:
for name in ["02_clean.sql", "03_ledger.sql", "04_checks.sql", "05_reporting.sql"]:
    path = SQL / name
    if path.exists():
        run_sql_file(path)
    else:
        print("Skipped", name, "(not created yet)")

Ran 02_clean.sql - 4 batches
Ran 03_ledger.sql - 5 batches
Ran 04_checks.sql - 4 batches
Ran 05_reporting.sql - 2 batches


In [6]:
summary = pd.read_sql("""
    SELECT 'raw.requests' AS object, COUNT(*) AS rows FROM raw.requests
    UNION ALL SELECT 'clean.requests', COUNT(*) FROM clean.requests
    UNION ALL SELECT 'clean.handle_time', COUNT(*) FROM clean.handle_time
    UNION ALL SELECT 'clean.roster', COUNT(*) FROM clean.roster
    UNION ALL SELECT 'raw.events', COUNT(*) FROM raw.events
""", engine)
display(summary)
display(pd.read_sql("SELECT * FROM cap.settings", engine))

,object,rows
0,raw.requests,1272
1,clean.requests,1272
2,clean.handle_time,12
3,clean.roster,113
4,raw.events,80


,paid_hours,shrinkage,ramp_1_4,ramp_5_8,ramp_9_12,hourly_cost
0,37.5,0.22,0.25,0.5,0.75,30.0
